# Load, Group and Clean Data

In [11]:
import json
from collections import defaultdict

def load_reviews(file_path="data\\raw\\Tcomments.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    table = next(item for item in data if item.get("type") == "table")
    data = table["data"]

    return data

def build_final_structure(reviews, ids_file_path):
    # Load ids.json
    with open(ids_file_path, "r", encoding="utf-8") as f:
        ids_data = json.load(f)
    
    table = next(item for item in ids_data if item.get("type") == "table")
    ids_data = table["data"]

    # Create lookup dictionary from ids.json
    # Key = id, Value = full metadata
    ids_lookup = {item["id"]: item for item in ids_data}

    # Group reviews by idquestion
    grouped_reviews = defaultdict(list)
    for review in reviews:
        grouped_reviews[review["idquestion"]].append({
            "content": review["content"],
            "rank": review["rank"],   # keep as string (as requested)
            "time": review["time"]
        })

    # Build final list
    final_output = []

    for idquestion, reviews_list in grouped_reviews.items():
        if idquestion not in ids_lookup:
            continue  # skip if no matching id in ids.json

        id_info = ids_lookup[idquestion]

        final_output.append({
            "course_id": id_info["id"],
            "course_name": id_info["course"],
            "lecturer": id_info["lecture"],
            "reviews": reviews_list,
            "metadata": {
                "tag": id_info["tag"],
                "num_reviews": len(reviews_list)
            }
        })

    return final_output

import re

class Cleaner:
    def __init__(self, text):
        self.text = text


    def clean_hebrew_text(self, text):
        if not isinstance(text, str):
            return ""

        # 1. Normalize whitespace
        text = text.replace("\xa0", " ")
        text = text.replace("\t", " ")
        text = text.replace("\r", " ")
        text = re.sub(r"\s+", " ", text).strip()

        # 2. Normalize punctuation
        text = text.replace("...", "…")
        text = text.replace("..", "…")
        text = re.sub(r"!{2,}", "!", text)
        text = re.sub(r"\?{2,}", "?", text)
        text = text.replace(" - ", " — ")

        # 3. Remove noise + invisible chars
        text = text.replace("\\/", "/")
        text = text.replace("\u200f", "")
        text = text.replace("\u200e", "")
        text = text.replace("\ufeff", "")
        text = re.sub(r"[\x00-\x1f]", "", text)

        # 4. Normalize Hebrew-specific chars
        text = text.replace("״", '"')
        text = text.replace("׳", "'")
        text = text.replace("־", "-")

        # 5. Clean HTML remnants
        text = re.sub(r"<[^>]+>", "", text)
        text = text.replace("&nbsp;", " ")
        text = text.replace("&quot;", '"')
        text = text.replace("&amp;", "&")

        # Final whitespace pass
        text = re.sub(r"\s+", " ", text).strip()

        return text

In [12]:
reviews = load_reviews()
grouped_data = build_final_structure(reviews, 'data\\raw\\Tquestions.json')

# Cleaning
cleaner = Cleaner("")
for course in grouped_data:
    for review in course["reviews"]:
        review["content"] = cleaner.clean_hebrew_text(review["content"])

# Save cleaned data
with open("data/cleaned_reviews.json", "w", encoding="utf-8") as f:
    json.dump(grouped_data, f, ensure_ascii=False, indent=2)

print("Cleaned data saved to data/cleaned_reviews.json")

Cleaned data saved to data/cleaned_reviews.json


In [13]:
print(grouped_data[0])

{'course_id': '86', 'course_name': 'מבוא לביולוגיה חישובית', 'lecturer': 'רחל', 'reviews': [{'content': 'ביולוגיה חישובית אני לקחתי בשנה שעברה, מאוד אהבתי, אבל אני אוהבת ביולוגיה. זה קורס באלגוריתמים בתכלס, עם תכנות במטלב. המבחן במועד א היה ממש סבבה, מועד ב לא עביר. בסהכ זה קורס מעניין.', 'rank': '0', 'time': '2013-02-26 15:51:14'}], 'metadata': {'tag': 'choose', 'num_reviews': 1}}
